# Test Llama Stack Distribution

This notebook tests the deployed LlamaStackDistribution instance to verify:
1. Connection to Llama Stack server
2. LLM inference via vLLM
3. Tool execution via MCP (pg-airman-mcp)
4. Agent orchestration

## Prerequisites

- LlamaStackDistribution deployed via `make install PROVIDER_MODE=llama_stack`
- Route accessible at `llama-stack-api-{namespace}.apps...`

## 1. Setup and Configuration

In [1]:
# Install llama-stack-client version 0.3.5 to match server version
!pip install llama-stack-client==0.3.5 -q

In [2]:
import os
import json
import urllib3
from llama_stack_client import LlamaStackClient

# Disable SSL warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Configuration
NAMESPACE = "samouelian-dev"  # Change to your namespace
LLAMA_STACK_URL = f"https://llama-stack-api-{NAMESPACE}.apps.ai-dev02.kni.syseng.devcluster.openshift.com"

print(f"Connecting to Llama Stack at: {LLAMA_STACK_URL}")

Connecting to Llama Stack at: https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com


## 2. Initialize Client

In [3]:
# Initialize client
client = LlamaStackClient(base_url=LLAMA_STACK_URL)

print("✓ Client initialized")
print(f"Base URL: {client.base_url}")

✓ Client initialized
Base URL: https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com


## 3. List Available Models

In [4]:
# List available models
try:
    models_response = client.models.list()
    
    print("=== Available Models ===")
    if hasattr(models_response, 'data'):
        models = models_response.data
    else:
        models = models_response if isinstance(models_response, list) else []
    
    print(f"Found {len(models)} model(s):\n")
    for model in models:
        print(f"Model ID: {model.identifier}")
        print(f"  Provider: {model.provider_id}")
        print(f"  Resource ID: {model.provider_resource_id}")
        print(f"  Type: {model.model_type}")
        print()
    
    # Store the full model identifier for use in chat
    if models:
        FULL_MODEL_ID = models[0].identifier
        print(f"✓ Will use model: {FULL_MODEL_ID}")
    else:
        print("❌ No models found!")
        
except Exception as e:
    print(f"❌ Error listing models: {e}")
    import traceback
    traceback.print_exc()

INFO:httpx:HTTP Request: GET https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/models "HTTP/1.1 200 OK"


=== Available Models ===
Found 1 model(s):

Model ID: vllm-inference/redhataillama-31-8b-instruct
  Provider: vllm-inference
  Resource ID: redhataillama-31-8b-instruct
  Type: llm

✓ Will use model: vllm-inference/redhataillama-31-8b-instruct


## 4. Check Providers

In [5]:
# Get available providers
try:
    providers_response = client.providers.list()
    
    print("=== Available Providers ===")
    
    # Handle both list and response object formats
    if isinstance(providers_response, list):
        providers = providers_response
    elif hasattr(providers_response, 'data'):
        providers = providers_response.data
    else:
        providers = [providers_response]
    
    for provider in providers:
        print(f"\n{provider.api.upper()} API:")
        print(f"  Provider ID: {provider.provider_id}")
        print(f"  Type: {provider.provider_type}")
        
        # Show config (mask sensitive data)
        if hasattr(provider, 'config') and provider.config:
            config_display = dict(provider.config)
            if 'api_token' in config_display:
                config_display['api_token'] = '********'
            print(f"  Config: {json.dumps(config_display, indent=4, default=str)}")
        
except Exception as e:
    print(f"❌ Error getting providers: {e}")
    import traceback
    traceback.print_exc()

INFO:httpx:HTTP Request: GET https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/providers "HTTP/1.1 200 OK"


=== Available Providers ===

INFERENCE API:
  Provider ID: vllm-inference
  Type: remote::vllm
  Config: {
    "url": "https://redhataillama-31-8b-instruct-quickstarts-llm.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1",
    "api_token": "********",
    "model": "vllm-inference/redhataillama-31-8b-instruct"
}

AGENTS API:
  Provider ID: meta-agents
  Type: inline::meta-reference
  Config: {
    "inference_model": "vllm-inference/redhataillama-31-8b-instruct",
    "persistence": {
        "agent_state": {
            "backend": "local-kv-store",
            "namespace": "default",
            "table_name": "agent_state"
        },
        "responses": {
            "backend": "local-sql-store",
            "namespace": "default",
            "table_name": "responses"
        }
    }
}

SAFETY API:
  Provider ID: llama-guard
  Type: inline::llama-guard

VECTOR_IO API:
  Provider ID: faiss
  Type: inline::faiss
  Config: {
    "persistence": {
        "backend": "local-kv-store",
  

## 5. Test Basic Inference

In [6]:
# Test basic chat completion
print("Testing basic inference...\n")

try:
    model_to_use = FULL_MODEL_ID if 'FULL_MODEL_ID' in globals() else "vllm-inference/redhataillama-31-8b-instruct"
    
    print(f"Using model: {model_to_use}\n")
    
    response = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {
                "role": "user",
                "content": "What is 2+2? Answer in one sentence."
            }
        ],
        stream=False,
    )
    
    print("Response:")
    if hasattr(response, 'choices') and response.choices:
        message = response.choices[0].message
        print(f"  Role: {message.role}")
        print(f"  Content: {message.content}")
    else:
        print(response)
    
    print("\n✓ Basic inference working!")
    
except Exception as e:
    print(f"❌ Error during inference: {e}")
    import traceback
    traceback.print_exc()

Testing basic inference...

Using model: vllm-inference/redhataillama-31-8b-instruct



INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/chat/completions "HTTP/1.1 200 OK"


Response:
  Role: assistant
  Content: The result of the equation 2+2 is 4.

✓ Basic inference working!


## 6. Register MCP Tools as a Tool Group

**CRITICAL STEP**: MCP tools must be registered as a toolgroup before they can be used by the LLM!

In [7]:
# Register MCP tools as a toolgroup
print("=== Registering MCP Tool Group ===\n")

try:
    # Get the MCP endpoint from provider configuration
    providers = client.providers.list()
    if isinstance(providers, list):
        tool_provider = next((p for p in providers if p.api == 'tool_runtime'), None)
    else:
        tool_provider = next((p for p in providers.data if p.api == 'tool_runtime'), None)
    
    if not tool_provider:
        raise Exception("Tool runtime provider not found!")
    
    mcp_uri = tool_provider.config.get('mcp_endpoint', {}).get('uri')
    provider_id = tool_provider.provider_id
    print(f"MCP Endpoint: {mcp_uri}")
    print(f"Provider ID: {provider_id}")
    print(f"Provider Type: {tool_provider.provider_type}")
    
    # Register the MCP toolgroup
    print("\nRegistering toolgroup...")
    print(f"  Using provider_id='{provider_id}' (from Llama Stack config)")
    
    try:
        response = client.toolgroups.register(
            toolgroup_id="mcp::pg_airman",
            provider_id=provider_id,
            mcp_endpoint={"uri": mcp_uri}
        )
        
        print(f"\n✓ Toolgroup registered successfully!")
        print(f"  Toolgroup ID: mcp::pg_airman")
        print(f"  Provider: {provider_id}")
        
    except Exception as e:
        error_msg = str(e)
        if "already exists" in error_msg.lower() or "duplicate" in error_msg.lower():
            print(f"\n✓ Toolgroup already registered (this is OK)")
        else:
            raise
    
    # List tools after registration
    print("\nListing tools after registration...")
    tools_response = client.tool_runtime.list_tools()
    
    if isinstance(tools_response, list):
        tools = tools_response
    elif hasattr(tools_response, 'data'):
        tools = tools_response.data
    else:
        tools = []
    
    print(f"Found {len(tools)} tool(s):")
    for tool in tools:
        tool_name = tool.name if hasattr(tool, 'name') else str(tool)
        tool_desc = tool.description if hasattr(tool, 'description') else 'N/A'
        print(f"\n  ✓ {tool_name}")
        print(f"    {tool_desc}")
    
except Exception as e:
    print(f"❌ Error registering toolgroup: {e}")
    import traceback
    traceback.print_exc()

INFO:httpx:HTTP Request: GET https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/providers "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/toolgroups "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/tool-runtime/list-tools "HTTP/1.1 200 OK"


=== Registering MCP Tool Group ===

MCP Endpoint: http://pg-airman-mcp-service.samouelian-dev.svc.cluster.local:8000/sse
Provider ID: mcp-tools
Provider Type: remote::model-context-protocol

Registering toolgroup...
  Using provider_id='mcp-tools' (from Llama Stack config)

✓ Toolgroup registered successfully!
  Toolgroup ID: mcp::pg_airman
  Provider: mcp-tools

Listing tools after registration...
Found 10 tool(s):

  ✓ list_schemas
    List all schemas in the database

  ✓ list_objects
    List objects in a schema with comments

  ✓ get_object_details
    Show detailed information about a database object with comments

  ✓ explain_query
    Explains the execution plan for a SQL query, showing how the database will execute it and provides detailed cost estimates.

  ✓ analyze_workload_indexes
    Analyze frequently executed queries in the database and recommend optimal indexes

  ✓ analyze_query_indexes
    Analyze a list of (up to 10) SQL queries and recommend optimal indexes

  ✓ an

## 7. Test Tool Calling with MCP (Agents API)

**Note:** Registered toolgroups (like MCP servers) must be used through the Agents API in llama-stack 0.3.5, not directly with chat completions.

In [8]:
# Test tool calling with MCP via Agents API
print("Testing tool calling with registered MCP toolgroup...\n")

try:
    model_to_use = FULL_MODEL_ID if 'FULL_MODEL_ID' in globals() else "vllm-inference/redhataillama-31-8b-instruct"
    
    print(f"Using model: {model_to_use}")
    print(f"Using toolgroup: mcp::pg_airman\n")
    
    # Check if agents API is available
    print("Checking for Agents API...")
    if hasattr(client, 'agents'):
        print("  ✓ Found: client.agents")
        agents_api = client.agents
    elif hasattr(client, 'alpha') and hasattr(client.alpha, 'agents'):
        print("  ✓ Found: client.alpha.agents")
        agents_api = client.alpha.agents
    else:
        raise Exception("Agents API not found in client")
    
    # Create agent with proper configuration
    print("Creating agent...")
    agent_response = agents_api.create(
        agent_config={
            "model": model_to_use,
            "instructions": "You are a helpful database assistant. Use the available tools to query the database and answer questions.",
            "toolgroups": ["mcp::pg_airman"],
            "tool_choice": "auto",
            "input_shields": [],
            "output_shields": [],
            "max_infer_iters": 10,
            "sampling_params": {
                "max_tokens": 2048,
                "temperature": 0.7,
                "top_p": 0.95,
            },
        }
    )
    agent_id = agent_response.agent_id
    print(f"✓ Agent created: {agent_id}\n")
    
    # Create session
    print("Creating session...")
    session_response = agents_api.session.create(
        agent_id=agent_id,
        session_name="test-session"
    )
    session_id = session_response.session_id
    print(f"✓ Session created: {session_id}\n")
    
    # Execute turn
    print("Executing query: 'List the first 3 tables in the database.'\n")
    response_stream = agents_api.turn.create(
        agent_id=agent_id,
        session_id=session_id,
        messages=[
            {
                "role": "user",
                "content": "List the first 3 tables in the database."
            }
        ],
        stream=True
    )
    
    # Process streaming response with debug output
    print("=" * 70)
    print("EVENT STREAM (showing all events):")
    print("=" * 70)
    
    event_count = 0
    tool_calls_made = []
    tool_responses = []
    final_response = None
    
    for chunk in response_stream:
        event_count += 1
        
        if hasattr(chunk, 'event') and hasattr(chunk.event, 'payload'):
            payload = chunk.event.payload
            event_type = payload.event_type if hasattr(payload, 'event_type') else 'unknown'
            
            # Print event type for debugging
            print(f"\n[Event #{event_count}: {event_type}]", end='')
            
            # Step progress - streaming text (suppress verbose output, just show we got it)
            if event_type == 'step_progress':
                if hasattr(payload, 'text_delta_model_response'):
                    print(" (text streaming...)", end='')
            
            # Step complete - tool calls or responses
            elif event_type == 'step_complete':
                if hasattr(payload, 'step_details'):
                    details = payload.step_details
                    
                    # Tool calls
                    if hasattr(details, 'tool_calls') and details.tool_calls:
                        print(f"\n  → Tool calls detected: {len(details.tool_calls)}")
                        for tc in details.tool_calls:
                            print(f"     • {tc.tool_name}({tc.arguments})")
                            tool_calls_made.append({'name': tc.tool_name, 'args': tc.arguments})
                    
                    # Tool responses
                    if hasattr(details, 'tool_responses') and details.tool_responses:
                        print(f"\n  → Tool responses received: {len(details.tool_responses)}")
                        for tr in details.tool_responses:
                            result_preview = str(tr.content)[:100]
                            print(f"     • {tr.tool_name}: {result_preview}...")
                            tool_responses.append({'name': tr.tool_name, 'result': tr.content})
            
            # Turn complete - final answer
            elif event_type == 'turn_complete':
                print(f"\n  → Turn completed!")
                if hasattr(payload, 'turn'):
                    turn = payload.turn
                    if hasattr(turn, 'output_message'):
                        msg = turn.output_message
                        if hasattr(msg, 'content'):
                            final_response = msg.content
                            print(f"     • Final response length: {len(final_response)} chars")
        else:
            print(f"\n[Event #{event_count}: Unknown chunk type]")
    
    print("\n" + "=" * 70)
    print(f"SUMMARY: Processed {event_count} events")
    print("=" * 70)
    
    # Print results
    if tool_calls_made:
        print(f"\n✓ Tool calls made ({len(tool_calls_made)}):")
        for tc in tool_calls_made:
            print(f"  - {tc['name']}({tc['args']})")
    
    if tool_responses:
        print(f"\n✓ Tool responses received ({len(tool_responses)}):")
        for tr in tool_responses:
            print(f"  - {tr['name']}: {str(tr['result'])[:200]}...")
    
    if final_response:
        print(f"\n✓ Final Answer:")
        print("-" * 70)
        print(final_response)
        print("-" * 70)
    else:
        print("\n⚠ WARNING: No final answer received!")
        print("  This may indicate:")
        print("  - Stream ended before turn completed")
        print("  - Tool execution failed")
        print("  - Agent reached max_infer_iters without completing")
    
    print("\n" + "=" * 70)
    print("✓ Tool calling test completed!")
    
except Exception as e:
    print(f"❌ Error during tool calling: {e}")
    import traceback
    traceback.print_exc()

INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1alpha/agents "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1alpha/agents/8732b823-e98e-4096-967a-fd9947668ed2/session "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1alpha/agents/8732b823-e98e-4096-967a-fd9947668ed2/session/c49b66e4-9cd8-4f7b-88bc-fbfdb2f228e8/turn "HTTP/1.1 200 OK"


Testing tool calling with registered MCP toolgroup...

Using model: vllm-inference/redhataillama-31-8b-instruct
Using toolgroup: mcp::pg_airman

Checking for Agents API...
  ✓ Found: client.alpha.agents
Creating agent...
✓ Agent created: 8732b823-e98e-4096-967a-fd9947668ed2

Creating session...
✓ Session created: c49b66e4-9cd8-4f7b-88bc-fbfdb2f228e8

Executing query: 'List the first 3 tables in the database.'

EVENT STREAM (showing all events):

[Event #1: step_start]
[Event #2: step_progress]
[Event #3: step_progress]
[Event #4: step_progress]
[Event #5: step_progress]
[Event #6: step_progress]
[Event #7: step_progress]
[Event #8: step_progress]
[Event #9: step_progress]
[Event #10: step_progress]
[Event #11: step_progress]
[Event #12: step_complete]
[Event #13: step_start]
[Event #14: step_progress]
[Event #15: step_complete]
  → Tool calls detected: 1
     • list_objects({"object_type": "table", "schema_name": "public", "limit": "3"})

  → Tool responses received: 1
     • list_obje

## 8. Test Streaming Inference

In [9]:
print("Testing streaming inference...\n")

try:
    model_to_use = FULL_MODEL_ID if 'FULL_MODEL_ID' in globals() else "vllm-inference/redhataillama-31-8b-instruct"
    
    print(f"Using model: {model_to_use}\n")
    print("Streaming response:")
    
    stream = client.chat.completions.create(
        model=model_to_use,
        messages=[
            {
                "role": "user",
                "content": "Count from 1 to 5, one number per line."
            }
        ],
        stream=True,
    )
    
    for chunk in stream:
        if hasattr(chunk, 'choices') and chunk.choices:
            delta = chunk.choices[0].delta
            if hasattr(delta, 'content') and delta.content:
                print(delta.content, end='', flush=True)
    
    print("\n\n✓ Streaming working!")
    
except Exception as e:
    print(f"❌ Error during streaming: {e}")
    import traceback
    traceback.print_exc()

INFO:httpx:HTTP Request: POST https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com/v1/chat/completions "HTTP/1.1 200 OK"


Testing streaming inference...

Using model: vllm-inference/redhataillama-31-8b-instruct

Streaming response:
1
2
3
4
5

✓ Streaming working!


## 9. Summary

In [10]:
print("=" * 70)
print("LLAMA STACK DEPLOYMENT TEST SUMMARY")
print("=" * 70)
print(f"\nEndpoint: {LLAMA_STACK_URL}")
print(f"Model: {FULL_MODEL_ID if 'FULL_MODEL_ID' in globals() else 'vllm-inference/redhataillama-31-8b-instruct'}")
print(f"Namespace: {NAMESPACE}")
print("\nTests completed:")
print("  ✓ Connection to Llama Stack")
print("  ✓ Model listing (client.models.list())")
print("  ✓ Provider configuration (client.providers.list())")
print("  ✓ Basic inference (client.chat.completions.create())")
print("  ✓ MCP toolgroup registration (client.toolgroups.register())")
print("  ✓ Tool calling via Agents API (client.agents.create/turn.create)")
print("  ✓ Streaming responses")
print("\n" + "=" * 70)
print("CRITICAL FINDINGS:")
print("")
print("1. Model Identifier:")
print("   ✓ Correct:   'vllm-inference/redhataillama-31-8b-instruct'")
print("   ✗ Incorrect: 'redhataillama-31-8b-instruct'")
print("")
print("2. MCP Tool Registration:")
print("   ✓ MUST use provider_id from config (e.g., 'mcp-tools')")
print("   ✗ NOT generic type like 'model-context-protocol'")
print("   ✓ Dynamically extract provider_id: tool_provider.provider_id")
print("   ✓ MUST register toolgroup: client.toolgroups.register()")
print("   ✓ MUST use Agents API for registered toolgroups")
print("   ✗ chat.completions.create() does NOT support tool_groups parameter!")
print("")
print("3. MCP Service Name:")
print("   ✓ Service name is 'pg-airman-mcp-service' (with -service suffix)")
print("   ✗ NOT 'pg-airman-mcp' (without suffix)")
print("   ✓ Update values.yaml: serviceName: 'pg-airman-mcp-service'")
print("")
print("4. MCP Transport Configuration:")
print("   ✓ Llama Stack 0.3.5 requires SSE transport (/sse endpoint)")
print("   ✗ Streamable HTTP (/mcp endpoint) not yet supported")
print("   ✓ Single pg-airman-mcp instance with transport set via PROVIDER_MODE")
print("   → PROVIDER_MODE=llama_stack → transport=sse (endpoint: /sse)")
print("   → PROVIDER_MODE=mcp_direct → transport=streamable-http (endpoint: /mcp)")
print("=" * 70)
print("\nIf all tests passed, the Llama Stack deployment is ready for")
print("integration with the copilot backend!")
print("\nKey integration requirements:")
print("  1. Use full model IDs with provider prefix")
print("  2. Deploy with PROVIDER_MODE=llama_stack for SSE transport")
print("  3. Use correct service name: pg-airman-mcp-service")
print("  4. Extract provider_id dynamically from configuration")
print("  5. Use Agents API (NOT chat completions) for MCP toolgroups")
print("  6. Register toolgroups on backend startup")

LLAMA STACK DEPLOYMENT TEST SUMMARY

Endpoint: https://llama-stack-api-samouelian-dev.apps.ai-dev02.kni.syseng.devcluster.openshift.com
Model: vllm-inference/redhataillama-31-8b-instruct
Namespace: samouelian-dev

Tests completed:
  ✓ Connection to Llama Stack
  ✓ Model listing (client.models.list())
  ✓ Provider configuration (client.providers.list())
  ✓ Basic inference (client.chat.completions.create())
  ✓ MCP toolgroup registration (client.toolgroups.register())
  ✓ Tool calling via Agents API (client.agents.create/turn.create)
  ✓ Streaming responses

CRITICAL FINDINGS:

1. Model Identifier:
   ✓ Correct:   'vllm-inference/redhataillama-31-8b-instruct'
   ✗ Incorrect: 'redhataillama-31-8b-instruct'

2. MCP Tool Registration:
   ✓ MUST use provider_id from config (e.g., 'mcp-tools')
   ✗ NOT generic type like 'model-context-protocol'
   ✓ Dynamically extract provider_id: tool_provider.provider_id
   ✓ MUST register toolgroup: client.toolgroups.register()
   ✓ MUST use Agents API f